# Improving earthquake metadata for GNSS time series analysis using Okada-based co-seismic deformation modelling
# ***Point Source Model***

## 0. Workflow

Read all Global CMT solutions $\to$ Apply Okada 1985 Model [1] $\to$ Compute displacements for every earthquake, for every station.

*** 

[1] Y. Okada, “Surface deformation due to shear and tensile faults in a half-space,” Bulletin of the Seismological Society of America, vol. 75, no. 4, pp. 1135–1154, Aug. 1985. doi:10.1785/bssa0750041135

## 1. Global CMT

Global Centroid Moment Tensor solutions [2,3]

Download the .ndk file of CMT solutions from 1976 to 2020 from: **https://www.globalcmt.org/CMTfiles.html**

There should be exactly $56,832$ earthquakes **jan76_dec20.ndk**.

***

[2] A. M. Dziewonski, T.-A. Chou, and J. H. Woodhouse, “Determination of earthquake source parameters from waveform data for studies of global and regional seismicity,” Journal of Geophysical Research, vol. 86, pp. 2825–2852, 1981, doi: 10.1029/JB086iB04p02825.

[3] G. Ekström, M. Nettles, and A. M. Dziewonski, “The global CMT project 2004–2010: Centroid-moment tensors for 13,017 earthquakes,” Physics of the Earth and Planetary Interiors, vols. 200–201, pp. 1–9, 2012, doi: 10.1016/j.pepi.2012.04.002.

In [1]:
from obspy import read_events

full_catalog = read_events("jan76_dec20.ndk")

print(str(len(full_catalog)))

56832


However, as we will see later, the Okada 1985 model uses a planar approximation. Therefore, we must only collect earthquakes in the vicinity of Greece, that is: 34-42 N, 19-29 E. These are mapped in  **Thesis_Mapping.ipynb**.

The remaining events should be 496.

In [2]:
from obspy import UTCDateTime
import time

start_time = UTCDateTime("1997-01-01T00:00:00")
end_time   = UTCDateTime("2021-01-01T00:00:00")

catalog = full_catalog.filter(
    "time >= %s" % start_time,
    "time <= %s" % end_time,
    "magnitude >= 4.5",
    "latitude >= 34",
    "latitude <= 42",
    "longitude >= 19",
    "longitude <= 29"
)

catalog.write("Greece_Seismicity.xml", format="QUAKEML")
print(len(catalog))

496


If we examine the contents of this catalog, we will find the origin of the events in lon, lat, dep, as well as the moment tensor and nodal planes.

Note that the CMT does not distinguish between fault and auxiliary planes. This is only possible through fault geology and aftershock studies. This poses a great problem in modelling finite source models where the orientation of the fault is essential to the model.

Fortunately, for point source models, the displacement field resulting from either nodal plane is identical. Therefore we shall use the point-source computation from Okada 1985 [1].

In [3]:
for event in catalog:
    # Extract preferred origin information
    origin = event.preferred_origin() or event.origins[0]
    
    # Extract focal mechanism / moment tensor info
    focal_mech = event.preferred_focal_mechanism() or event.focal_mechanisms[0]
    tensor = focal_mech.moment_tensor.tensor

    # Extract Nodal Planes (Strike, Dip, Rake)
    nodal_planes = focal_mech.nodal_planes
    plane1 = nodal_planes.nodal_plane_1
    
    # Extra check: sometimes Nodal Plane 2 might not be populated, so safely grab it
    plane2 = getattr(nodal_planes, 'nodal_plane_2', None)
    
    print(f"Event ID: {event.resource_id.id.split('/')[-1]}")
    print(f"Time: {origin.time}, Lat: {origin.latitude}, Lon: {origin.longitude}, Depth: {origin.depth/1000} km")
    print(f"Moment Tensor components: Mrr={tensor.m_rr}, Mtt={tensor.m_tt}, Mpp={tensor.m_pp}\n")
    print(f"Plane 1 -> Strike: {plane1.strike}°, Dip: {plane1.dip}°, Rake: {plane1.rake}°")
    print(f"Plane 2 -> Strike: {plane2.strike}°, Dip: {plane2.dip}°, Rake: {plane2.rake}°")
    
    break

Event ID: event
Time: 1997-05-16T07:00:54.000000Z, Lat: 41.06, Lon: 20.14, Depth: 20.5 km
Moment Tensor components: Mrr=-7.49e+16, Mtt=-3.65e+16, Mpp=1.114e+17

Plane 1 -> Strike: 213.0°, Dip: 34.0°, Rake: -40.0°
Plane 2 -> Strike: 338.0°, Dip: 69.0°, Rake: -117.0°


## 2. Okada 1985 : Point Source Equations & Function Implementation

In [7]:
import numpy as np
from obspy.geodetics import gps2dist_azimuth

The displacement field $u_i(x_1,x_2,x_3)$ due to a dislocation $\Delta u_j(\xi_1.\xi_2,\xi_3)$ across an internal surface $\Sigma$ in an isotropic medium is given by:

\begin{equation}
u_i=\frac{1}{F}\int\int_\Sigma\Delta u_j\biggr[\lambda\delta_{jk}\frac{\partial u_i^j}{\partial\xi_k}+\mu\biggr(\frac{\partial u_i^j}{\partial\xi_k}+\frac{\partial u_i^k}{\partial\xi_j}\biggr)\biggr]\nu_kd\Sigma \tag{1}
\end{equation}

where $\delta_{jk}$ is the Kronecker delta, $\lambda$ and $\mu$ are Lame parameters, $\nu_k$ is the direction cosine of the normal to the surface element $d\Sigma$, and the summation convention applies. $u_i^j$ is the $i$th component of the displacement at $(x_1,x_2,x_3)$ due to the $j$th direction point force of magnitude $F$ at $(\xi_1,\xi_2\xi_3)$ [4].

Okada [1] provides a compact solution for this.

***

[4] J. A. Steketee, “Some geophysical applications of the elasticity theory of dislocations,” Can. J. Phys., vol. 36, no. 9, pp. 1168–1197, Sep. 1958. [Online]. Available: Canadian Science Publishing

Note however that $U$ is the amount of slip and $\Delta\Sigma$ is the rupture area. Using these conventions, we can compute the moment of an eathquake to be:

\begin{equation}
M_0=U\cdot\Delta\Sigma\cdot\mu
\end{equation}

Wherein $\mu$ is the shear modulus. From this, we can compute $U\cdot\Delta\Sigma$ to be $M_0/\mu$. This will be used in the program.

### 2.1 Translation into Okada's Coordinate System

In [30]:
def coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, opening, U, sim):
    delsigma=1.0
    if (sim==True): #This is used for the simulations given in Okada 1985 Table 2, for simplicity
        dEast = stalon - evlon
        dNorth = stalat - evlat
    else: # This is used for accurate distance computation of real-life coordinates
        distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, stalat, stalon)
        distance_km = distance_m / 1000.0
        dEast = distance_km*np.sin(azimuth*np.pi/180.0)
        dNorth = distance_km*np.cos(azimuth*np.pi/180.0)
    return(
        delsigma, #km #delsigma
        U*np.cos(rake*np.pi/180.0), #km #U1
        U*np.sin(rake*np.pi/180.0), #km #U2
        opening, #km #U3
        dEast * np.sin(strike*np.pi/180.0) + dNorth * np.cos(strike*np.pi/180.0), #km #x
        -dEast * np.cos(strike*np.pi/180.0) + dNorth * np.sin(strike*np.pi/180.0), #km #y
        abs(evdep), #km #d
        30.0, #GPa #mu
        30.0, #GPa #lam
        dip*np.pi/180.0 #in radians #delta
    )

### 2.2 Displacements

For Strike-Slip:
\begin{cases}
u_x^0=-\frac{U_1}{2\pi}\biggr[\frac{3x^2q}{R^5}+I_1^0\sin\delta\biggr]\Delta\Sigma \\
u_y^0=-\frac{U_1}{2\pi}\biggr[\frac{3xyq}{R^5}+I_2^0\sin\delta\biggr]\Delta\Sigma \\
u_z^0=-\frac{U_1}{2\pi}\biggr[\frac{3xdq}{R^5}+I_4^0\sin\delta\biggr]\Delta\Sigma \\
\tag{2}
\end{cases}

For Dip-Slip:
\begin{cases}
u_x^0=-\frac{U_2}{2\pi}\biggr[\frac{3xpq}{R^5}-I_3^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
u_y^0=-\frac{U_2}{2\pi}\biggr[\frac{3ypq}{R^5}-I_1^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
u_z^0=-\frac{U_2}{2\pi}\biggr[\frac{3dpq}{R^5}-I_5^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
\tag{3}
\end{cases}

For Tensile Fault:
\begin{cases}
u_x^0=\frac{U_3}{2\pi}\biggr[\frac{3xq^2}{R^5}-I_3^0\sin^2\delta\biggr]\Delta\Sigma \\
u_y^0=\frac{U_3}{2\pi}\biggr[\frac{3y^2q}{R^5}-I_1^0\sin^2\delta\biggr]\Delta\Sigma \\
u_z^0=\frac{U_3}{2\pi}\biggr[\frac{3dq^2}{R^5}-I_5^0\sin^2\delta\biggr]\Delta\Sigma \\
\tag{4}
\end{cases}

In [10]:
def disp_strike (U1, x, y, d, q, R, I1, I2, I4, delta, delsigma):
    return (
        -U1/(2.0*np.pi)*((3.0*x**2.0*q/R**5.0)+I1*np.sin(delta))*delsigma,
        -U1/(2.0*np.pi)*((3.0*x*y*q/R**5.0)+I2*np.sin(delta))*delsigma,
        -U1/(2.0*np.pi)*((3.0*x*d*q/R**5.0)+I4*np.sin(delta))*delsigma
    )

def disp_dip (U2, x, y, d, p, q, R, I1, I3, I5, delta, delsigma):
    return (
        -U2/(2.0*np.pi)*((3.0*x*p*q/R**5.0)-I3*np.sin(delta)*np.cos(delta))*delsigma,
        -U2/(2.0*np.pi)*((3.0*y*p*q/R**5.0)-I1*np.sin(delta)*np.cos(delta))*delsigma,
        -U2/(2.0*np.pi)*((3.0*d*p*q/R**5.0)-I5*np.sin(delta)*np.cos(delta))*delsigma
    )

def disp_tens (U3, x, y, d, q, R, I1, I3, I5, delta, delsigma):
    return (
        U3/(2.0*np.pi)*((3.0*x*q**2.0/R**5.0)-I3*np.sin(delta)**2.0)*delsigma,
        U3/(2.0*np.pi)*((3.0*y*q**2.0/R**5.0)-I1*np.sin(delta)**2.0)*delsigma,
        U3/(2.0*np.pi)*((3.0*d*q**2.0/R**5.0)-I5*np.sin(delta)**2.0)*delsigma 
    )

Wherein the intermediate variables are:

\begin{cases}
I_1^0=\frac{\mu}{\lambda+\mu}y\biggr[\frac{1}{R(R+d)^2}-x^2\frac{3R+d}{R^3(R+d)^3}\biggr]\\
I_2^0=\frac{\mu}{\lambda+\mu}x\biggr[\frac{1}{R(R+d)^2}-y^2\frac{3R+d}{R^3(R+d)^3}\biggr]\\
I_3^0=\frac{\mu}{\lambda+\mu}\biggr[\frac{x}{R^3}\biggr]-I_2^0\\
I_4^0=\frac{\mu}{\lambda+\mu}\biggr[-xy\frac{2R+d}{R^3(R+d)^2}\biggr]\\
I_5^0=\frac{\mu}{\lambda+\mu}\biggr[\frac{1}{R(R+d)}-x^2\frac{2R+d}{R^3(R+d)^2}\biggr]
\tag{5}
\end{cases}

\begin{cases}
p=y\cos\delta+d\sin\delta\\
q=y\sin\delta-d\cos\delta\\
R^2=x^2+y^2+d^2=x^2+p^2+q^2
\tag{6}
\end{cases}

In [4]:
def pqrs (x, y, d, delta):
    p = y*np.cos(delta)+d*np.sin(delta)
    q = y*np.sin(delta)-d*np.cos(delta)
    return (
        p, #p
        q, #q
        np.sqrt(x**2.0+y**2.0+d**2.0), #R
        p*np.sin(delta)+q*np.cos(delta) #s
    )

def disp_I (x, y, d, R, mu, lam):
    I2=(mu/(lam+mu))*x*((1.0/(R*(R+d)**2.0))-y**2.0*(3.0*R+d)/(R**3.0*(R+d)**3.0))
    return(
        (mu/(lam+mu))*y*((1.0/(R*(R+d)**2.0))-x**2.0*(3.0*R+d)/(R**3.0*(R+d)**3.0)),
        I2,
        (mu/(lam+mu))*(x/R**3.0)-I2,
        (mu/(lam+mu))*(-x*y*(2.0*R+d)/(R**3.0*(R+d)**2.0)),
        (mu/(lam+mu))*((1.0/(R*(R+d)))-x**2.0*(2.0*R+d)/(R**3.0*(R+d)**2.0))
    )

Putting it all together:

In [19]:
def displacement (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim) :
    delsigma, U1, U2, U3, x, y, d, mu, lam, delta = coords(stalon, stalat, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
    p, q, R, s = pqrs (x, y, d, delta)
    I1, I2, I3, I4, I5 = disp_I (x, y, d, R, mu, lam)
    ux_strike, uy_strike, uz_strike = disp_strike (U1, x, y, d, q, R, I1, I2, I4, delta, delsigma)
    ux_dip, uy_dip, uz_dip = disp_dip (U2, x, y, d, p, q, R, I1, I3, I5, delta, delsigma)
    ux_tens, uy_tens, uz_tens = disp_tens (U3, x, y, d, q, R, I1, I3, I5, delta, delsigma)
    ux = ux_strike + ux_dip + ux_tens
    uy = uy_strike + uy_dip + uy_tens
    uz = uz_strike + uz_dip + uz_tens
    return (ux, uy, uz)

Testing with Table 2 of Okada 1985:

Recall that the rake angles are:
- Normal Fault: -90 deg
- Reverse/ Thrust: 90 deg
- Strike Slip: 0 or 180 deg

In [21]:
stalon, stalat, stadep, evlon, evlat, evdep, sim = -3.0, 2.0, 0.0, 0.0, 0.0, 4.0, True #x=2, y=3, d=4
strike, dip = 0.0, 70.0 #degrees

print("Strike-Slip Results: -9.447E-4, -1.023E-3, -7.420E-4")
rake, opening, U = 0.0, 0.0, 1.0
ux, uy, uz = displacement (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("ux={:0.3e}".format(ux))
print("uy={:0.3e}".format(uy))
print("uz={:0.3e}".format(uz))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Dip-Slip Results: -1.17E-3, -2.082E-3, -2.532E-3")
rake, opening, U = 90.0, 0.0, 1.0
ux, uy, uz = displacement (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("ux={:0.3e}".format(ux))
print("uy={:0.3e}".format(uy))
print("uz={:0.3e}".format(uz))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Tensile Results: -3.572E-4, +3.531E-4, -2.007E-4")
rake, opening, U = 0.0, 1.0, 0.0
ux, uy, uz = displacement (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("ux={:0.3e}".format(ux))
print("uy={:0.3e}".format(uy))
print("uz={:0.3e}".format(uz))

Strike-Slip Results: -9.447E-4, -1.023E-3, -7.420E-4
ux=-9.447e-04
uy=-1.023e-03
uz=-7.420e-04
- - - - - - - - - - - - - - - - - - - - - - - - - -
Dip-Slip Results: -1.17E-3, -2.082E-3, -2.532E-3
ux=-1.172e-03
uy=-2.082e-03
uz=-2.532e-03
- - - - - - - - - - - - - - - - - - - - - - - - - -
Tensile Results: -3.572E-4, +3.531E-4, -2.007E-4
ux=-3.572e-04
uy=3.531e-04
uz=-2.007e-04


### 2.3 Strains

For Strike-Slip:
\begin{cases}
\frac{\partial u_x^0}{\partial x}=-\frac{U_1}{2\pi}\biggr[\frac{3xq}{R^5}\biggr(2-\frac{5x^2}{R^2}\biggr)+J_1^0\sin\delta\biggr]\Delta\Sigma \\
\frac{\partial u_x^0}{\partial y}=-\frac{U_1}{2\pi}\biggr[-\frac{15x^2yq}{R^7}+\biggr(\frac{3x^2}{R^5}+J_2^0\biggr)\sin\delta\biggr]\Delta\Sigma \\
\frac{\partial u_y^0}{\partial x}=-\frac{U_1}{2\pi}\biggr[\frac{3yq}{R^5}\biggr(1-\frac{5x^2}{R^2}\biggr)+J_2^0\sin\delta\biggr]\Delta\Sigma \\
\frac{\partial u_y^0}{\partial y}=-\frac{U_1}{2\pi}\biggr[\frac{3xq}{R^5}\biggr(1-\frac{5y^2}{R^2}\biggr)+\biggr(\frac{3xy}{R^5}+J_4^0\biggr)\sin\delta\biggr]\Delta\Sigma \\
\tag{7}
\end{cases}

For Dip-Slip:
\begin{cases}
\frac{\partial u_x^0}{\partial x}=-\frac{U_2}{2\pi}\biggr[\frac{3pq}{R^5}\biggr(1-\frac{5x^2}{R^2}\biggr)-J_3^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
\frac{\partial u_x^0}{\partial y}=-\frac{U_2}{2\pi}\biggr[\frac{3x}{R^5}\biggr(s-\frac{5ypq}{R^2}\biggr)-J_1^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
\frac{\partial u_y^0}{\partial x}=-\frac{U_2}{2\pi}\biggr[-\frac{15xypq}{R^7}-J_1^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
\frac{\partial u_y^0}{\partial y}=-\frac{U_2}{2\pi}\biggr[\frac{3pq}{R^5}\biggr(1-\frac{5y^2}{R^2}\biggr)+\frac{3ys}{R^5}-J_2^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
\tag{8}
\end{cases}

For Tensile Fault:
\begin{cases}
\frac{\partial u_x^0}{\partial x}=\frac{U_3}{2\pi}\biggr[\frac{3q^2}{R^5}\biggr(1-\frac{5x^2}{R^2}\biggr)-J_3^0\sin^2\delta\biggr]\Delta\Sigma \\
\frac{\partial u_x^0}{\partial y}=\frac{U_3}{2\pi}\biggr[\frac{3xq}{R^5}\biggr(2\sin\delta-\frac{5yq}{R^2}\biggr)-J_1^0\sin^2\delta\biggr]\Delta\Sigma \\
\frac{\partial u_y^0}{\partial x}=\frac{U_3}{2\pi}\biggr[-\frac{15xyq^2}{R^7}-J_1^0\sin^2\delta\biggr]\Delta\Sigma \\
\frac{\partial u_y^0}{\partial y}=\frac{U_3}{2\pi}\biggr[\frac{3q}{R^5}\biggr(q+2y\sin\delta-\frac{5y^2q}{R^2}\biggr)-J_2^0\sin^2\delta\biggr]\Delta\Sigma \\
\tag{9}
\end{cases}

In [22]:
def strain_strike (U1, x, y, d, q, R, J1, J2, J4, delta, delsigma):
    return (
        -U1/(2.0*np.pi)*((3.0*x*q/R**5.0)*(2.0-(5.0*x**2.0/R**2.0))+J1*np.sin(delta))*delsigma,
        -U1/(2.0*np.pi)*(-(15.0*x**2.0*y*q/R**7.0)+((3.0*x**2.0/R**5.0)+J2)*np.sin(delta))*delsigma,
        -U1/(2.0*np.pi)*((3.0*y*q/R**5.0)*(1.0-(5.0*x**2.0/R**2.0))+J2*np.sin(delta))*delsigma,
        -U1/(2.0*np.pi)*((3.0*x*q/R**5.0)*(1.0-(5.0*y**2.0/R**2.0))+((3.0*x*y/R**5)+J4)*np.sin(delta))*delsigma  
    )

def strain_dip (U2, x, y, d, p, q, s, R, J1, J2, J3, delta, delsigma):
    return (
        -U2/(2.0*np.pi)*((3.0*p*q/R**5.0)*(1.0-(5.0*x**2.0/R**2.0))-J3*np.sin(delta)*np.cos(delta))*delsigma,
        -U2/(2.0*np.pi)*((3.0*x/R**5.0)*(s-(5.0*y*p*q/R**2.0))-J1*np.sin(delta)*np.cos(delta))*delsigma,
        -U2/(2.0*np.pi)*(-(15.0*x*y*p*q/R**7)-J1*np.sin(delta)*np.cos(delta))*delsigma,
        -U2/(2.0*np.pi)*((3.0*p*q/R**5.0)*(1.0-(5.0*y**2.0/R**2.0))+(3.0*y*s/R**5.0)-J2*np.sin(delta)*np.cos(delta))*delsigma 
    )

def strain_tens (U3, x, y, d, q, R, J1, J2, J3, delta, delsigma):
    return (
        U3/(2.0*np.pi)*((3.0*q**2/R**5.0)*(1.0-(5.0*x**2/R**2.0))-J3*(np.sin(delta))**2.0)*delsigma,
        U3/(2.0*np.pi)*((3.0*x*q/R**5.0)*(2.0*np.sin(delta)-(5*y*q/R**2))-J1*(np.sin(delta))**2.0)*delsigma,
        U3/(2.0*np.pi)*(-(15.0*x*y*q**2/R**7)-J1*(np.sin(delta))**2.0)*delsigma,
        U3/(2.0*np.pi)*((3.0*q/R**5.0)*(q+2.0*y*np.sin(delta)-(5.0*y**2.0*q/R**2.0))-J2*(np.sin(delta))**2.0)*delsigma        
    )

Wherein the intermediate variables are:

\begin{cases}
J_1^0=\frac{\mu}{\lambda+\mu}\biggr[-3xy\frac{3R+d}{R^3(R+d)^3}+3x^3y\frac{5R^2+4Rd+d^2}{R^5(R+d)^4}\biggr]\\
J_2^0=\frac{\mu}{\lambda+\mu}\biggr[\frac{1}{R^3}-\frac{3}{R(R+d)^2}+3x^2y^2\frac{5R^2+4Rd+d^2}{R^5(R+d)^4}\biggr]\\
J_3^0=\frac{\mu}{\lambda+\mu}\biggr[\frac{1}{R^3}-\frac{3x^2}{R^5}\biggr]-J_2^0\\
J_4^0=\frac{\mu}{\lambda+\mu}\biggr[-\frac{3xy}{R^5}\biggr]-J_1^0
\tag{10}
\end{cases}


\begin{cases}
p=y\cos\delta+d\sin\delta\\
q=y\sin\delta-d\cos\delta\\
s=p\sin\delta+q\cos\delta\\
R^2=x^2+y^2+d^2=x^2+p^2+q^2
\tag{11}
\end{cases}

In [23]:
def strain_J (x, y, d, R, mu, lam):
    J1 = (mu/(lam+mu))*(-3.0*x*y*(3.0*R+d)/(R**3.0*(R+d)**3.0)+3.0*x**3.0*y*(5.0*R**2.0+4.0*R*d+d**2)/(R**5.0*(R+d)**4.0))
    J2 = (mu/(lam+mu))*((1.0/R**3.0)-3.0/(R*(R+d)**2.0)+3.0*x**2.0*y**2.0*(5.0*R**2.0+4.0*R*d+d**2)/(R**5.0*(R+d)**4.0))
    return(
        J1,
        J2,
        (mu/(lam+mu))*((1.0/R**3.0)-(3.0*x**2.0/R**5.0))-J2,
        (mu/(lam+mu))*(-(3.0*x*y/R**5.0))-J1
    )

Putting it all together:

In [24]:
def strain (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim) :
    delsigma, U1, U2, U3, x, y, d, mu, lam, delta = coords(stalon, stalat, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
    p, q, R, s = pqrs (x, y, d, delta)
    J1, J2, J3, J4 = strain_J (x, y, d, R, mu, lam)
    uxx_strike, uxy_strike, uyx_strike, uyy_strike = strain_strike (U1, x, y, d, q, R, J1, J2, J4, delta, delsigma)
    uxx_dip, uxy_dip, uyx_dip, uyy_dip = strain_dip (U2, x, y, d, p, q, s, R, J1, J2, J3, delta, delsigma)
    uxx_tens, uxy_tens, uyx_tens, uyy_tens = strain_tens (U3, x, y, d, q, R, J1, J2, J3, delta, delsigma)  
    uxx = uxx_strike + uxx_dip + uxx_tens
    uxy = uxy_strike + uxy_dip + uxy_tens
    uyx = uyx_strike + uyx_dip + uyx_tens
    uyy = uyy_strike + uyy_dip + uyy_tens
    return (uxx, uxy, uyx, uyy)

Testing with Table 2 of Okada 1985:

In [25]:
stalon, stalat, stadep, evlon, evlat, evdep, sim = -3.0, 2.0, 0.0, 0.0, 0.0, 4.0, True #x=2, y=3, d=4
strike, dip = 0.0, 70.0 #degrees

print("Strike-Slip Results: -2.286E-4, -1.425E-4, -2.051E-4, 3.007E-4")
rake, opening, U = 0.0, 0.0, 1.0
uxx, uxy, uyx, uyy = strain (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("uxx={:0.3e}".format(uxx))
print("uxy={:0.3e}".format(uxy))
print("uyx={:0.3e}".format(uyx))
print("uyy={:0.3e}".format(uyy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Dip-Slip Results: -1.526E-4, -3.544E-4, +6.983E-4, -1.154E-3")
rake, opening, U = 90.0, 0.0, 1.0
uxx, uxy, uyx, uyy = strain (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("uxx={:0.3e}".format(uxx))
print("uxy={:0.3e}".format(uxy))
print("uyx={:0.3e}".format(uyx))
print("uyy={:0.3e}".format(uyy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Tensile Results: -1.360E-4, +5.073E-4, -6.773E-5, +6.811E-4")
rake, opening, U = 0.0, 1.0, 0.0
uxx, uxy, uyx, uyy = strain (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("uxx={:0.3e}".format(uxx))
print("uxy={:0.3e}".format(uxy))
print("uyx={:0.3e}".format(uyx))
print("uyy={:0.3e}".format(uyy))

Strike-Slip Results: -2.286E-4, -1.425E-4, -2.051E-4, 3.007E-4
uxx=-2.286e-04
uxy=-1.425e-04
uyx=-2.051e-04
uyy=-3.007e-04
- - - - - - - - - - - - - - - - - - - - - - - - - -
Dip-Slip Results: -1.526E-4, -3.544E-4, +6.983E-4, -1.154E-3
uxx=-1.526e-04
uxy=-3.544e-04
uyx=6.983e-04
uyy=-1.154e-03
- - - - - - - - - - - - - - - - - - - - - - - - - -
Tensile Results: -1.360E-4, +5.073E-4, -6.773E-5, +6.811E-4
uxx=-1.360e-04
uxy=5.073e-04
uyx=-6.773e-05
uyy=6.811e-04


### 2.4 Tilts

For Strike-Slip:
\begin{cases}
    \frac{\partial u_z^0}{\partial x}=-\frac{U_1}{2\pi}\biggr[\frac{3dq}{R^5}\biggr(1-\frac{5x^2}{R^2}\biggr)+K_1^0\sin\delta\biggr]\Delta\Sigma \\
    \frac{\partial u_z^0}{\partial y}=-\frac{U_1}{2\pi}\biggr[-\frac{15xydq}{R^7}+\biggr(\frac{3xd}{R^5}+K_2^0\biggr)\sin\delta\biggr]\Delta\Sigma
\tag{12}
\end{cases}

For Dip-Slip:
\begin{cases}
    \frac{\partial u_z^0}{\partial x}=-\frac{U_2}{2\pi}\biggr[-\frac{15xdpq}{R^7}-K_3^0\sin\delta\cos\delta\biggr]\Delta\Sigma \\
    \frac{\partial u_z^0}{\partial y}=-\frac{U_2}{2\pi}\biggr[\frac{3d}{R^5}\biggr(s-\frac{5ypq}{R^2}\biggr)-K_1^0\sin\delta\cos\delta\biggr]\Delta\Sigma
\tag{13}
\end{cases}

For Tensile Fault:
\begin{cases}
    \frac{\partial u_z^0}{\partial x}=\frac{U_3}{2\pi}\biggr[-\frac{15xdq^2}{R^7}-K_3^0\sin^2\delta\biggr]\Delta\Sigma \\
    \frac{\partial u_z^0}{\partial y}=\frac{U_3}{2\pi}\biggr[\frac{3dq}{R^5}\biggr(2\sin\delta-\frac{5yq}{R^2}\biggr)-K_1^0\sin^2\delta\biggr]\Delta\Sigma
\tag{14}
\end{cases}

In [26]:
def tilt_strike (U1, x, y, d, q, R, K1, K2, delta, delsigma):
    return (
        -U1/(2.0*np.pi)*((3.0*d*q/R**5)*(1.0-(5.0*x**2/R**2))+K1*np.sin(delta))*delsigma,
        -U1/(2.0*np.pi)*(-(15.0*x*y*d*q/R**7.0)+((3.0*x*d/R**5.0)+K2)*np.sin(delta))*delsigma
    )

def tilt_dip (U2, x, y, d, p, q, R, s, K1, K3, delta, delsigma):
    return (
        -U2/(2.0*np.pi)*(-(15.0*x*d*p*q/R**7.0)-K3*np.sin(delta)*np.cos(delta))*delsigma,
        -U2/(2.0*np.pi)*((3.0*d/R**5.0)*(s-(5.0*y*p*q/R**2.0))-K1*np.sin(delta)*np.cos(delta))*delsigma
    )

def tilt_tens (U3, x, y, d, q, R, K1, K3, delta, delsigma):
    return (
        U3/(2.0*np.pi)*(-(15.0*x*d*q**2.0/R**7.0)-K3*(np.sin(delta))**2.0)*delsigma,
        U3/(2.0*np.pi)*((3.0*d*q/R**5.0)*(2.0*np.sin(delta)-(5.0*y*q/R**2.0))-K1*(np.sin(delta))**2.0)*delsigma
    )

\begin{cases}
K_1^0=-\frac{\mu}{\lambda+\mu}y\biggr[\frac{2R+d}{R^3(R+d)^2}-x^2\frac{8R^2+9Rd+3d^2}{R^5(R+d)^3}\biggr]\\
K_2^0=-\frac{\mu}{\lambda+\mu}x\biggr[\frac{2R+d}{R^3(R+d)^2}-y^2\frac{8R^2+9Rd+3d^2}{R^5(R+d)^3}\biggr]\\
K_3^0=-\frac{\mu}{\lambda+\mu}\biggr[\frac{3xd}{R^5}\biggr]-K_2^0\\
\tag{15}
\end{cases}

\begin{cases}
p=y\cos\delta+d\sin\delta\\
q=y\sin\delta-d\cos\delta\\
R^2=x^2+y^2+d^2=x^2+p^2+q^2
\tag{16}
\end{cases}

In [27]:
def tilt_K (x, y, d, R, mu, lam):
    K2 = -mu/(lam+mu)*x*((2.0*R+d)/(R**3.0*(R+d)**2.0)-y**2.0*(8.0*R**2.0+9.0*R*d+3.0*d**2.0)/(R**5.0*(R+d)**3.0))
    return(
        -mu/(lam+mu)*y*((2.0*R+d)/(R**3.0*(R+d)**2.0)-x**2.0*(8.0*R**2.0+9.0*R*d+3.0*d**2.0)/(R**5.0*(R+d)**3.0)),
        K2,
        -mu/(lam+mu)*((3.0*x*d)/(R**5.0))-K2
    )

Putting it all together

In [28]:
def tilt (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim) :
    delsigma, U1, U2, U3, x, y, d, mu, lam, delta = coords(stalon, stalat, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
    p, q, R, s = pqrs (x, y, d, delta)
    K1, K2, K3 = tilt_K (x, y, d, R, mu, lam)
    uzx_strike, uzy_strike = tilt_strike (U1, x, y, d, q, R, K1, K2, delta, delsigma)
    uzx_dip, uzy_dip = tilt_dip (U2, x, y, d, p, q, R, s, K1, K3, delta, delsigma)
    uzx_tens, uzy_tens = tilt_tens (U3, x, y, d, q, R, K1, K3, delta, delsigma)
    uzx = uzx_strike + uzx_dip + uzx_tens
    uzy = uzy_strike + uzy_dip + uzy_tens
    return (uzx, uzy)

Testing with Table 2 of Okada 1985:

In [29]:
stalon, stalat, stadep, evlon, evlat, evdep, sim = -3.0, 2.0, 0.0, 0.0, 0.0, 4.0, True #x=2, y=3, d=4

print("Strike-Slip Results: -6.259E-5, -1.693E-4")
rake, opening, U = 0.0, 0.0, 1.0
uzx, uzy = tilt (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("uzx={:0.3e}".format(uzx))
print("uzy={:0.3e}".format(uzy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Dip-Slip Results: +8.707E-4, -6.345E-4")
rake, opening, U = 90.0, 0.0, 1.0
uzx, uzy = tilt (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("uzx={:0.3e}".format(uzx))
print("uzy={:0.3e}".format(uzy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Tensile Results: +7.541E-5, +8.104E-4")
rake, opening, U = 0.0, 1.0, 0.0
uzx, uzy = tilt (stalon, stalat, stadep, evlon, evlat, evdep, strike, dip, rake, opening, U, sim)
print("uzx={:0.3e}".format(uzx))
print("uzy={:0.3e}".format(uzy))

Strike-Slip Results: -6.259E-5, -1.693E-4
uzx=-6.259e-05
uzy=-1.693e-04
- - - - - - - - - - - - - - - - - - - - - - - - - -
Dip-Slip Results: +8.707E-4, -6.345E-4
uzx=8.707e-04
uzy=-6.345e-04
- - - - - - - - - - - - - - - - - - - - - - - - - -
Tensile Results: +7.541E-5, +8.104E-4
uzx=7.541e-05
uzy=8.104e-04


## 3. Application to Samos 2020 Mw 7.0 Earthquake

In [36]:
from obspy import read_events

### 3.1 Acquiring Event Magnitude and Location

In [39]:
# SAMOS ISLAND EARTHQUAKE
#https://geofon.gfz.de/eqinfo/event.php?id=gfz2020vimx

catalog = read_events("Greece_Seismicity.xml", format="QUAKEML")

max_event = max(catalog, key=lambda event: event.preferred_magnitude().mag)
mag = max_event.preferred_magnitude().mag
origin = max_event.preferred_origin() or max_event.origins
print(f"Maximum Magnitude: {mag}")
print(f"Time: {origin.time}, Location: ({origin.latitude}, {origin.longitude})")    

Maximum Magnitude: 6.99
Time: 2020-10-30T11:51:34.800000Z, Location: (37.78, 26.63)


### 3.2 Determining Relevant Stations
According to [5], the IGS procedure for determining the radius relevant to an earthquake with respect to the Okada 1985 model is to use the magnitude. Specifically:

\begin{equation}
    r = 10^{0.5M — 0.8} [km]
\tag{17}
\end{equation}

wherein $r$ is the relevant radius, and $M$ is the magnitude of the earthquake.

***
[5] E. Serpelloni, A. Cavaliere, L. Martelli, F. Pintori, L. Anderlini, A. Borghi, D. Randazzo, S. Bruni, R. Devoti, P. Perfetti, and S. Cacciaguerra, “Surface velocities and strain-rates in the Euro-Mediterranean region from massive GPS data processing,” Frontiers in Earth Science, vol. 10, Art. no. 907897, 2022, doi: 10.3389/feart.2022.907897.

In [41]:
import pandas as pd

In [65]:
evlat = origin.latitude
evlon = origin.longitude

radius = 10.0**(0.5*mag-0.8)
print(f"radius = {radius}")

df = pd.read_csv("NGL_Greece_Stations.csv", sep=",")

df_sta_ids = df["Station_ID"].astype(str)
df_sta_lats = pd.to_numeric(df["Latitude"], errors="coerce")
df_sta_lons = pd.to_numeric(df["Longitude"], errors="coerce")

sta_ids = []
sta_lats = []
sta_lons = []

new_df = pd.DataFrame()

for i in range(len(df_sta_ids)):
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, df_sta_lats[i], df_sta_lons[i])
    distance_km = distance_m / 1000.0
    if (distance_km <= radius):
        sta_ids.append(df_sta_ids[i])
        sta_lats.append(df_sta_lats[i])
        sta_lons.append(df_sta_lons[i])

new_df["Station_ID"] = sta_ids
new_df["Latitude"] = sta_lats
new_df["Longitude"] = sta_lons

new_df.to_csv("Samos_Stations.csv", index=False)

radius = 495.45019080479057


### 3.3 Computing Surface Deformation for Every Station

### 3.4 Plotting Contour Plot of Surface Deformation within Relevant Radius